In [ ]:
import pandas as pd

df_raw = pd.read_csv('/content/data.csv')

In [ ]:
!pip install NeuralForecast
!pip install Optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 52.2 MB/s eta 0:00:00
  Attempting uninstall: tornado
    Found existing installation: tornado 6.5.1
    Uninstalling tornado-6.5.1:
      Successfully uninstalled tornado-6.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is

In [ ]:
df = df_raw.copy()


In [ ]:

df['load_lag_48h'] = df['load'].shift(48)
df['load_lag_168h'] = df['load'].shift(168)
df['load_lag_168h'] = df['load'].shift(336)
df = df.dropna()

In [ ]:
df.set_index('timestamp', inplace=True)
df.index = pd.to_datetime(df.index)

# school holiday ranges
school_breaks = [
    # 2024
    ('2024-03-23', '2024-04-01'), # Easter 2024
    ('2024-05-09', '2024-05-12'), # Ascension Day 2024
    ('2024-05-18', '2024-05-20'), # Whitsun 2024
    ('2024-06-29', '2024-08-11'), # Summer 2024
    ('2024-10-12', '2024-10-20'), # Autumn 2024
    ('2024-12-21', '2025-01-05'), # Christmas 2024

    # 2025
    ('2025-02-08', '2025-02-16'), # Winter 2025
    ('2025-04-12', '2025-04-21'), # Easter 2025
    ('2025-05-29', '2025-06-01'), # Ascension Day 2025
    ('2025-06-07', '2025-06-09'), # Whitsun 2025
    ('2025-06-28', '2025-08-10'), # Summer 2025
    ('2025-10-11', '2025-10-19'), # Autumn 2025
    ('2025-12-23', '2026-01-04'), # Christmas 2025

    # 2026
    ('2026-02-07', '2026-02-15'), # Winter 2026
    ('2026-03-28', '2026-04-06')  # Easter 2026
]

# init flag
df['is_school_break'] = 0

# apply
tz = df.index.tz
for start, end in school_breaks:
    start_ts = pd.Timestamp(start, tz=tz)
    end_ts = pd.Timestamp(end, tz=tz) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

    df.loc[start_ts:end_ts, 'is_school_break'] = 1

# prev weak using shift
df['is_prev_week_school_break'] = df['is_school_break'].shift(168).fillna(0).astype(int)

In [ ]:
df['Total_Wind'] = df['Wind Offshore'] +  df['Wind Onshore']

In [ ]:
import numpy as np

df['Wind_Log'] = np.log1p(df['Total_Wind'])

# interaction terms
df['Solar_Temp'] = df['Solar'] * df['t2m']

noise_threshold = 0.1

# if physical solar generation exceeds the noise floor, keep it.
# otherwise, force it to absolute zero.
df['Solar_Gated'] = np.where(df['Solar'] > noise_threshold, df['Solar'], 0.0)

df['Solar_Log'] = np.log1p(df['Solar_Gated'])

In [ ]:
#-------------------------
#RIDGE REGRESSION FIRST
#-------------------------
import optuna
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler


features = [
    'tod_sin', 'tod_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos',
    'Total_Wind', 'Solar_Log', 'DT1', 'DT2', 'DT3',
    'TL2W_w', 'HDD',
    'load_lag_48h', 'load_lag_168h', 'is_school_break', 'is_prev_week_school_break'
]
target = 'load'

# drop n/a
df_clean = df.dropna(subset=features + [target]).copy()

df_clean = df_clean.sort_index()


test_duration = pd.Timedelta(days=56)
val_window = pd.Timedelta(days=7)
n_splits = int(test_duration / val_window)  

max_date = df_clean.index.max()
test_start_date = max_date - test_duration

def objective(trial):
    alpha = trial.suggest_float('alpha', 1e-4, 1000, log=True)

    fold_mses = []

    # Walk-Forward Loop
    for step in range(n_splits):
        val_start = test_start_date + (step * val_window)
        val_end = val_start + val_window

        train_mask = df_clean.index < val_start
        val_mask = (df_clean.index >= val_start) & (df_clean.index < val_end)

        X_train, y_train = df_clean.loc[train_mask, features], df_clean.loc[train_mask, target]
        X_val, y_val = df_clean.loc[val_mask, features], df_clean.loc[val_mask, target]

        #safeguard
        if len(X_val) == 0:
            continue

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        model = Ridge(alpha=alpha, random_state=42)
        model.fit(X_train_scaled, y_train)

        preds = model.predict(X_val_scaled)

        # Eval using MSE
        fold_mse = np.mean((y_val - preds) ** 2)
        fold_mses.append(fold_mse)

    return np.mean(fold_mses)

#optimization
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)

print("\n--- Optuna Study Completed ---")
print(f"Best Alpha: {study.best_params['alpha']:.5f}")

# =====================================================================
#                          EVALUATION
# =====================================================================
print("\nWalk-Forward CV with Best Alpha for Final Metrics...")

best_alpha = study.best_params['alpha']
all_y_true = []
all_y_pred = []

# collect raw predictions
for step in range(n_splits):
    val_start = test_start_date + (step * val_window)
    val_end = val_start + val_window

    train_mask = df_clean.index < val_start
    val_mask = (df_clean.index >= val_start) & (df_clean.index < val_end)

    X_train, y_train = df_clean.loc[train_mask, features], df_clean.loc[train_mask, target]
    X_val, y_val = df_clean.loc[val_mask, features], df_clean.loc[val_mask, target]

    if len(X_val) == 0:
        continue

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Initialize Ridge with the optimal alpha found by Optuna
    best_model = Ridge(alpha=best_alpha, random_state=42)
    best_model.fit(X_train_scaled, y_train)

    preds = best_model.predict(X_val_scaled)

    # Store actuals and predictions
    all_y_true.extend(y_val.values)
    all_y_pred.extend(preds)


ridge_metrics = evaluate_forecast(all_y_true, all_y_pred, "Ridge Baseline (56-Day TS-CV)")

# =====================================================================
#             Finalize the Model
# =====================================================================
print("\nTraining Final Scaled Model on full history...")
final_scaler = StandardScaler()
X_scaled_full = final_scaler.fit_transform(df_clean[features])

final_ridge = Ridge(alpha=best_alpha, random_state=42)
final_ridge.fit(X_scaled_full, df_clean[target])

[I 2026-05-20 16:43:14,998] A new study created in memory with name: no-name-86b21ae0-0558-48b1-85d3-f548e8246e92
[I 2026-05-20 16:43:15,163] Trial 0 finished with value: 62009.04088281144 and parameters: {'alpha': 0.7880866464205896}. Best is trial 0 with value: 62009.04088281144.


Starting Walk-Forward Optuna Study (8 splits of 7 days)...


[I 2026-05-20 16:43:15,303] Trial 1 finished with value: 62010.14389636477 and parameters: {'alpha': 0.0004875138506795899}. Best is trial 0 with value: 62009.04088281144.
[I 2026-05-20 16:43:15,440] Trial 2 finished with value: 62001.63650184263 and parameters: {'alpha': 6.297003145549122}. Best is trial 2 with value: 62001.63650184263.
[I 2026-05-20 16:43:15,576] Trial 3 finished with value: 61952.31092436033 and parameters: {'alpha': 119.97734421863451}. Best is trial 3 with value: 61952.31092436033.
[I 2026-05-20 16:43:15,716] Trial 4 finished with value: 61983.136804905014 and parameters: {'alpha': 22.27930564269558}. Best is trial 3 with value: 61952.31092436033.
[I 2026-05-20 16:43:15,855] Trial 5 finished with value: 61949.07958745251 and parameters: {'alpha': 105.54534912846997}. Best is trial 5 with value: 61949.07958745251.
[I 2026-05-20 16:43:16,009] Trial 6 finished with value: 62009.85308772835 and parameters: {'alpha': 0.20736225893963298}. Best is trial 5 with value: 61


--- Optuna Study Completed ---
Best Alpha: 85.84309

Re-running Walk-Forward CV with Best Alpha for Final Metrics...
--- Ridge Baseline (56-Day TS-CV) Final Evaluation ---
MAE:  174.64 MW
RMSE: 248.89 MW
MAPE: 5.346 %
-----------------------------------

Training Final Scaled Model on full history...
Final Ridge architecture is locked and ready for inference.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_forecast(y_true, y_pred, model_name="Model"):
    """
    Calculates MAE, RMSE, and MAPE.
    y_true: Pandas Series or NumPy array of actual load.
    y_pred: Pandas Series or NumPy array of predicted load.
    """
    
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # Epsilon to prevent division by zero
    epsilon = 1e-10
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + epsilon))) * 100

    # Results
    print(f"--- {model_name} Final Evaluation ---")
    print(f"MAE:  {mae:.2f} MW")
    print(f"RMSE: {rmse:.2f} MW")
    print(f"MAPE: {mape:.3f} %")
    print("-" * 35)

    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

In [ ]:
#---------------------------------
#XGBoost (Global)
#---------------------------------

from xgboost import XGBRegressor


df['hour'] = df.index.hour
df['dayofweek'] = df.index.dayofweek
df['month'] = df.index.month

# feature set for Trees
features = [
    'hour', 'dayofweek', 'month',
    'Total_Wind', 'Solar_Log', 'DT1', 'DT2', 'DT3',
    'TL2W_w', 'HDD',
    'load_lag_48h', 'load_lag_168h',
    'is_school_break', 'is_prev_week_school_break'
]
target = 'load'

df_clean = df.dropna(subset=features + [target]).copy()
df_clean = df_clean.sort_index()

# TS-CV Setup
test_duration = pd.Timedelta(days=56)
val_window = pd.Timedelta(days=7)
n_splits = int(test_duration / val_window)  # 8 splits

max_date = df_clean.index.max()
test_start_date = max_date - test_duration

# Optuna Objective
def objective(trial):
    #XGBoost params
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'tree_method': 'hist',
        'random_state': 42,
        'n_jobs': -1 
    }

    fold_mses = []

    for step in range(n_splits):
        val_start = test_start_date + (step * val_window)
        val_end = val_start + val_window

        train_mask = df_clean.index < val_start
        val_mask = (df_clean.index >= val_start) & (df_clean.index < val_end)

        X_train, y_train = df_clean.loc[train_mask, features], df_clean.loc[train_mask, target]
        X_val, y_val = df_clean.loc[val_mask, features], df_clean.loc[val_mask, target]

        if len(X_val) == 0:
            continue

        # Fit
        model = XGBRegressor(**params)
        model.fit(X_train, y_train, verbose=False)

        preds = model.predict(X_val)

        # Calculate objective metric (MSE)
        fold_mse = mean_squared_error(y_val, preds)
        fold_mses.append(fold_mse)

    return np.mean(fold_mses)

# Start Optuna
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)

print("\n--- Optuna Study Completed ---")
print(f"Best MSE: {study.best_value:.3f}")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# --- Evaluate
print("\nWalk-Forward CV with Best Parameters to extract Final Metrics...")

all_y_true = []
all_y_pred = []
best_params = study.best_params.copy()
best_params.update({'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1})

for step in range(n_splits):
    val_start = test_start_date + (step * val_window)
    val_end = val_start + val_window

    train_mask = df_clean.index < val_start
    val_mask = (df_clean.index >= val_start) & (df_clean.index < val_end)

    X_train, y_train = df_clean.loc[train_mask, features], df_clean.loc[train_mask, target]
    X_val, y_val = df_clean.loc[val_mask, features], df_clean.loc[val_mask, target]

    if len(X_val) == 0:
        continue

    best_model = XGBRegressor(**best_params)
    best_model.fit(X_train, y_train, verbose=False)

    preds = best_model.predict(X_val)

    all_y_true.extend(y_val.values)
    all_y_pred.extend(preds)

# Call eval function
xgb_metrics = evaluate_forecast(all_y_true, all_y_pred, "XGBoost Global (56-Day TS-CV)")

# --- Train Final Global XGBoost Model
final_xgb = XGBRegressor(**best_params)
final_xgb.fit(df_clean[features], df_clean[target])

[I 2026-05-20 16:53:11,689] A new study created in memory with name: no-name-0be51b22-5997-49c0-896e-6bedc8487313


Starting Global XGBoost Walk-Forward Optuna Study (8 splits of 7 days)...


[I 2026-05-20 16:54:03,968] Trial 0 finished with value: 55401.22004579988 and parameters: {'n_estimators': 1700, 'learning_rate': 0.016432144116928657, 'max_depth': 7, 'subsample': 0.6611922404629855, 'colsample_bytree': 0.6885616125060174}. Best is trial 0 with value: 55401.22004579988.
[I 2026-05-20 16:57:24,371] Trial 1 finished with value: 59076.27407561951 and parameters: {'n_estimators': 2300, 'learning_rate': 0.007073468615592818, 'max_depth': 10, 'subsample': 0.8936189525492346, 'colsample_bytree': 0.6007286356956968}. Best is trial 0 with value: 55401.22004579988.
[I 2026-05-20 16:58:42,372] Trial 2 finished with value: 57632.309753226895 and parameters: {'n_estimators': 2400, 'learning_rate': 0.039483930132183945, 'max_depth': 7, 'subsample': 0.6312636952864362, 'colsample_bytree': 0.7976918040089188}. Best is trial 0 with value: 55401.22004579988.
[I 2026-05-20 16:59:32,647] Trial 3 finished with value: 57119.238828736045 and parameters: {'n_estimators': 1700, 'learning_rat


--- Optuna Study Completed ---
Best MSE: 51242.652
  n_estimators: 3000
  learning_rate: 0.009560572925551623
  max_depth: 5
  subsample: 0.7291914455203549
  colsample_bytree: 0.6013634376245176

Re-running Walk-Forward CV with Best Parameters to extract Final Metrics...
--- XGBoost Global (56-Day TS-CV) Final Evaluation ---
MAE:  149.31 MW
RMSE: 226.37 MW
MAPE: 4.580 %
-----------------------------------

Training Final Global XGBoost Model on 100% of available history...
Final Global XGBoost model is locked and ready for Day-Ahead inference.


In [ ]:
#------------------------------
#XGBoost 24-Expert
#------------------------------


df['dayofweek'] = df.index.dayofweek
df['month'] = df.index.month

#Hour removed because it is a 24-expert model
features = [
    'dayofweek', 'month',
    'Total_Wind', 'Solar_Log', 'DT1', 'DT2', 'DT3',
    'TL2W_w', 'HDD',
    'load_lag_48h', 'load_lag_168h',
    'is_school_break', 'is_prev_week_school_break'
]
target = 'load'

df_clean = df.dropna(subset=features + [target]).copy()
df_clean = df_clean.sort_index()

# --- TS-CV Setup
test_duration = pd.Timedelta(days=56)
val_window = pd.Timedelta(days=7)
n_splits = int(test_duration / val_window)

max_date = df_clean.index.max()
test_start_date = max_date - test_duration


anchor_hours = [3, 8, 18]

#Objective Function
def objective(trial):
    # Different search space for 24-Expert model
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7), # Shallower to prevent overfitting
        'subsample': trial.suggest_float('subsample', 0.5, 0.8), # Aggressive subsampling
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'tree_method': 'hist',
        'random_state': 42,
        'n_jobs': -1
    }

    global_anchor_mses = []

    # Evaluate
    for anchor_h in anchor_hours:
        # Filter data for this specific hour
        df_hour = df_clean[df_clean.index.hour == anchor_h]

        hour_fold_mses = []

        # Start TS-CV
        for step in range(n_splits):
            val_start = test_start_date + (step * val_window)
            val_end = val_start + val_window

            train_mask = df_hour.index < val_start
            val_mask = (df_hour.index >= val_start) & (df_hour.index < val_end)

            X_train, y_train = df_hour.loc[train_mask, features], df_hour.loc[train_mask, target]
            X_val, y_val = df_hour.loc[val_mask, features], df_hour.loc[val_mask, target]

            # hour-specific 7-day validation
            if len(X_val) == 0:
                continue

            model = XGBRegressor(**params)
            model.fit(X_train, y_train, verbose=False)

            preds = model.predict(X_val)
            hour_fold_mses.append(mean_squared_error(y_val, preds))

        if hour_fold_mses:
            global_anchor_mses.append(np.mean(hour_fold_mses))

    # Final objective value = average MSE across the 3 anchor hours
    return np.mean(global_anchor_mses)

# --- Start Optuna
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print(f"Best Anchor Averaged MSE: {study.best_value:.3f}")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# --- Final 24 Experts
best_expert_params = study.best_params.copy()
best_expert_params.update({'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1})

# Dictionary for 24 distinct trained models
expert_models = {}

for h in range(24):
    df_h = df_clean[df_clean.index.hour == h]
    X_h, y_h = df_h[features], df_h[target]

    expert_xgb = XGBRegressor(**best_expert_params)
    expert_xgb.fit(X_h, y_h, verbose=False)

    expert_models[h] = expert_xgb

[I 2026-05-20 18:15:44,096] A new study created in memory with name: no-name-8d06a27a-176b-4d86-a4d8-09940725a730


Starting 24-Expert (Anchor Strategy) Optuna Study...


[I 2026-05-20 18:16:15,049] Trial 0 finished with value: 67159.60531399761 and parameters: {'n_estimators': 950, 'learning_rate': 0.08431564938804507, 'max_depth': 7, 'subsample': 0.5548121037476058, 'colsample_bytree': 0.5297714423808466}. Best is trial 0 with value: 67159.60531399761.
[I 2026-05-20 18:16:19,697] Trial 1 finished with value: 61778.833578937825 and parameters: {'n_estimators': 250, 'learning_rate': 0.14682627188688133, 'max_depth': 5, 'subsample': 0.6591789106519823, 'colsample_bytree': 0.7030574907112641}. Best is trial 1 with value: 61778.833578937825.
[I 2026-05-20 18:16:34,694] Trial 2 finished with value: 61780.4255662678 and parameters: {'n_estimators': 750, 'learning_rate': 0.09131804823642753, 'max_depth': 5, 'subsample': 0.5554994633946243, 'colsample_bytree': 0.7502762637638973}. Best is trial 1 with value: 61778.833578937825.
[I 2026-05-20 18:16:54,205] Trial 3 finished with value: 66606.14861310166 and parameters: {'n_estimators': 850, 'learning_rate': 0.10


--- Anchor Optimization Completed ---
Best Anchor Averaged MSE: 51040.205
  n_estimators: 350
  learning_rate: 0.05936429744280297
  max_depth: 3
  subsample: 0.6694301388745177
  colsample_bytree: 0.6067413311454889

Training Final 24 Expert Models on full history...
All 24 Expert XGBoost models are locked and ready for inference.


In [ ]:
!pip install neuralforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 75.6 MB/s eta 0:00:00
  Attempting uninstall: tornado
    Found existing installation: tornado 6.5.1
    Uninstalling tornado-6.5.1:
      Successfully uninstalled tornado-6.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires t

In [ ]:
df_uni = df.copy()

# ==========================================
# DATA PREP
# ==========================================
df_uni = df_uni.rename(columns={'load': 'y'})
if 'unique_id' not in df_uni.columns:
    df_uni['unique_id'] = 'DK1_Grid'
if 'ds' not in df_uni.columns:
    df_uni['ds'] = df_uni.index

df_uni['ds'] = pd.to_datetime(df_uni['ds'], utc=True).dt.tz_convert(None)

fut_exog_vars = [
    'tod_sin', 'tod_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'DT1', 'DT2', 'DT3',
    'Total_Wind', 'Solar_Log', 't2m',
]

strict_cols = ['unique_id', 'ds', 'y'] + fut_exog_vars
df_uni = df_uni[strict_cols].copy()

for col in fut_exog_vars + ['y']:
    df_uni[col] = pd.to_numeric(df_uni[col], errors='coerce').astype('float32')

df_uni = df_uni.dropna().sort_values(['unique_id', 'ds']).reset_index(drop=True)
print(f"Data Pipeline Cleared. Shape: {df_uni.shape}")

1. Initiating Sterile Data Pipeline & Feature Engineering...
Data Pipeline Cleared. Shape: (17208, 15)


In [ ]:
fut_exog_vars = [
    'Total_Wind', 'Solar_Log', 't2m',
]

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
import optuna
from neuralforecast.models import XLinear
from neuralforecast.losses.pytorch import MSE
from neuralforecast import NeuralForecast
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
# ==========================================
# TEMPORAL TRAIN/VAL SPLIT
# ==========================================
HORIZON = 36
val_days = 30
val_size = val_days * 24

last_date = df_uni['ds'].dt.date.max()

# Defining 23:00 cutoff to force an 11:00 AM cutoff internally
dam_cutoff_end = pd.to_datetime(f"{last_date} 23:00:00")
df_full_aligned = df_uni[df_uni['ds'] <= dam_cutoff_end].copy()

train_df = df_full_aligned.iloc[:-val_size].copy()
val_df = df_full_aligned.iloc[-val_size:].copy()

max_lookback = 336
inference_df = df_full_aligned.iloc[-(val_size + max_lookback):].copy()

# ==========================================
# 3. OPTUNA OBJECTIVE
# ==========================================
def objective(trial):
    input_size = trial.suggest_categorical('input_size', [168])
    hidden_size = trial.suggest_int('hidden_size', 64, 512, step=64)
    temporal_ff = trial.suggest_int('temporal_ff', 128, 512, step=64)
    channel_ff = trial.suggest_int('channel_ff', 8, 24, step=4)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128, 256])
    temporal_dropout = trial.suggest_float('temporal_dropout', 0.0, 0.4, step=0.1)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)

    model = XLinear(
        h=HORIZON,
        n_series=1,
        futr_exog_list=fut_exog_vars,
        input_size=input_size,
        hidden_size=hidden_size,
        temporal_ff=temporal_ff,
        channel_ff=channel_ff,
        temporal_dropout=temporal_dropout,
        loss=MSE(),
        learning_rate=learning_rate,
        batch_size=batch_size,
        max_steps=1000,
        scaler_type='standard',
        random_seed=42
    )

    nf = NeuralForecast(models=[model], freq='H')


    cv_df = nf.cross_validation(
        df=df_full_aligned,
        n_windows=7,
        step_size=24
    )
    cv_df['market_date'] = (cv_df['cutoff'] + pd.Timedelta(days=1)).dt.date
    market_cv_final = cv_df[cv_df['ds'].dt.date == cv_df['market_date']].copy()
    # Calculate MAPE
    mape = mean_absolute_percentage_error(market_cv_final['y'], market_cv_final['XLinear']) * 100



    return mape

In [ ]:


# ==========================================
# Start Study
# ==========================================


study = optuna.create_study(
    direction='minimize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

# Run 30 trials (Adjust depending on your Colab GPU limits)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("\n=========================================================")
print(f"Optimization Finished!")
print(f"Best Trial MAPE: {study.best_value:.3f}%")
print(f"Best Parameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")
print("=========================================================")

[I 2026-05-21 23:34:53,903] A new study created in memory with name: no-name-46517fff-8cc7-473c-9b68-c27e70db7be7



2. Initiating Optuna Study...


  0%|          | 0/30 [00:00<?, ?it/s]

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                 | Type          | Params | Mode 
---------------------------------------------------------------
0 | loss                 | MSE           | 0      | train
1 | padder_train         | ConstantPad1d | 0      | train
2 | scaler               | TemporalNorm  | 0      | train
3 | projection           | Sequential    | 129 K  | train
4 | temporal_gating      | GatingBlock   | 492 K  | train
5 | channel_gating       | GatingBlock   | 122    | train
6 | futr_exog_projection | Linear        | 428 K  | train
7 | head                 | Sequential    | 41.5 K | train
  | other params         | n/a           | 384    | n/a  
-------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1000` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

[I 2026-05-21 23:49:36,297] Trial 29 finished with value: 6.047013774514198 and parameters: {'input_size': 168, 'hidden_size': 192, 'temporal_ff': 384, 'channel_ff': 8, 'batch_size': 256, 'temporal_dropout': 0.2, 'learning_rate': 0.00011951545549992167}. Best is trial 21 with value: 4.10480722784996.

Optimization Finished!
Best Trial MAPE: 4.105%
Best Parameters:
    input_size: 336
    hidden_size: 64
    temporal_ff: 512
    channel_ff: 12
    batch_size: 256
    temporal_dropout: 0.0
    learning_rate: 0.0005735032530188406


In [ ]:
import pandas as pd

print("\n--- Top 10 Best Optuna Trials ---")

trials_df = study.trials_dataframe()

# filter for completed trials
completed_trials = trials_df[trials_df['state'] == 'COMPLETE']

# sort ascending
top_10_trials = completed_trials.sort_values(by='value', ascending=True).head(10)

param_cols = [col for col in top_10_trials.columns if col.startswith('params_')]
display_cols = ['number', 'value'] + param_cols

clean_df = top_10_trials[display_cols].rename(columns={'value': 'MSE', 'number': 'Trial'})

# output table
print(clean_df.to_string(index=False))


--- Top 10 Best Optuna Trials ---
 Trial      MSE  params_batch_size  params_channel_ff  params_hidden_size  params_input_size  params_learning_rate  params_temporal_dropout  params_temporal_ff
    21 4.104807                256                 12                  64                336              0.000574                      0.0                 512
    25 4.174995                128                 16                 256                336              0.000363                      0.0                 512
    17 4.207526                256                 12                 192                336              0.000586                      0.0                 448
    12 4.219126                 16                 16                  64                336              0.000953                      0.0                 448
     3 4.242720                256                 12                  64                336              0.000484                      0.0                 512
    2